# Phase 2 & 3: Data Cleaning, ETL Pipeline & Feature Engineering
**Project**: Unified Multi-Platform Social Media Analytics System
**Author**: Senior Data Analyst & Python Developer

In [ ]:
import os
import glob
import pandas as pd
import numpy as np

# Set base directory
base_dir = os.path.dirname(os.getcwd()) if 'python' in os.getcwd() else os.getcwd()
dataset_dir = os.path.join(base_dir, 'datasets')
print(f"Dataset directory: {dataset_dir}")

## 1. Load Raw Multi-Platform Datasets

In [ ]:
post_files = glob.glob(os.path.join(dataset_dir, '*_posts.csv'))
raw_df_list = []
for f in post_files:
    if 'cleaned' not in f:
        df_temp = pd.read_csv(f)
        print(f"Loaded {os.path.basename(f)}: {len(df_temp)} records")
        raw_df_list.append(df_temp)

df_combined = pd.concat(raw_df_list, ignore_index=True)
print(f"Combined Raw Posts Total: {len(df_combined)}")
df_combined.head()

## 2. Deduplication & Date Normalisation

In [ ]:
# Deduplication
initial_len = len(df_combined)
df_combined.drop_duplicates(subset=['post_id'], keep='first', inplace=True)
print(f"Removed {initial_len - len(df_combined)} duplicate posts.")

# Standardize mixed date strings to YYYY-MM-DD
df_combined['post_date'] = pd.to_datetime(df_combined['post_date'], format='mixed').dt.strftime('%Y-%m-%d')
df_combined['post_date'] = pd.to_datetime(df_combined['post_date'])
df_combined.info()

## 3. Imputation & NULL Value Handling

In [ ]:
df_combined['caption'] = df_combined['caption'].fillna('No Caption Provided')
df_combined['hashtags'] = df_combined['hashtags'].fillna('#General')

# Impute missing likes with platform & content_type median
df_combined['likes'] = df_combined.groupby(['platform', 'content_type'])['likes'].transform(lambda x: x.fillna(x.median()))

metric_cols = ['reach', 'impressions', 'comments', 'shares', 'saves', 'video_views', 'clicks']
df_combined[metric_cols] = df_combined[metric_cols].fillna(0).astype(int)
print("Missing values handled successfully.")

## 4. Feature Engineering & KPI Calculation

In [ ]:
# Time components
df_combined['posting_hour'] = pd.to_datetime(df_combined['post_time'], format='%H:%M:%S').dt.hour
df_combined['day_of_week'] = df_combined['post_date'].dt.day_name()
df_combined['day_num'] = df_combined['post_date'].dt.dayofweek
df_combined['is_weekend'] = df_combined['day_num'].isin([5, 6]).astype(int)
df_combined['month_year'] = df_combined['post_date'].dt.strftime('%Y-%m')

# Derived KPIs
df_combined['total_engagement'] = df_combined['likes'] + df_combined['comments'] + df_combined['shares'] + df_combined['saves']
df_combined['engagement_rate_pct'] = np.where(df_combined['reach'] > 0, (df_combined['total_engagement'] / df_combined['reach']) * 100, 0.0)
df_combined['ctr_pct'] = np.where(df_combined['impressions'] > 0, (df_combined['clicks'] / df_combined['impressions']) * 100, 0.0)
df_combined['virality_score'] = np.where(df_combined['reach'] > 0, (df_combined['shares'] / df_combined['reach']) * 100, 0.0)

# Save cleaned dataset
out_file = os.path.join(dataset_dir, 'cleaned_social_media_posts.csv')
df_combined.to_csv(out_file, index=False)
print(f"Saved cleaned dataset to {out_file}")